# 🔧 REAP Repair: Restoring Pruned MoE Models via Targeted Fine-Tuning

**A practical notebook for repairing capability loss in REAP-pruned (or otherwise expert-pruned) Mixture-of-Experts models.**

Authored by [Johnblick187](https://huggingface.co/Johnblick187) ("King Tweak") based on the workflow used to train [Opus-4.7-Reasoning-REAP-40B-A3B-LoRA-v2](https://huggingface.co/Johnblick187/Opus-4.7-Reasoning-REAP-40B-A3B-LoRA-v2).

---

## What is REAP and why does it need repair?

**REAP (Router-weighted Expert Activation Pruning)** is a technique that removes inactive or redundant experts from MoE models, dramatically shrinking model size. A 480B-parameter MoE can drop to 40B active params while preserving most capability.

**The problem:** Pruning leaves routing scars. The remaining experts now have to handle traffic that pruned siblings used to route. Math problems that activated 4 experts now have only 2 viable paths. Code reasoning that relied on niche experts gets routed to generalists. Performance degrades — sometimes catastrophically on edge cases.

**The fix:** Targeted fine-tuning on diverse, reasoning-heavy data that forces the surviving experts to recover lost specializations. This notebook walks through the full workflow.

---

## What this notebook covers

1. Environment setup and model loading (with optional LoRA continuation)
2. Dataset loading from HF (defaults to the REAP-Repair-Reasoning dataset)
3. Data preprocessing (handles the nested `<think>` issue + format standardization)
4. LoRA configuration tuned for MoE repair (with expert-LoRA on `gate_up_proj` / `down_proj`)
5. Training with safety rails (checkpoint cadence, OOM-safe configs)
6. Evaluation and HF push

## What you need

- A GPU with **at least 80GB VRAM** for 27-40B models (H100, H200, A100-80GB). RunPod recommended.
- A HuggingFace account with a write token
- ~$15-30 of compute budget for a full repair run (4-6 hours on H200)

## Honest expectations

- This won't fully restore an aggressively pruned model. ~50% expert pruning loses capability that 50K examples can't fully recover.
- It WILL meaningfully improve coherence, reasoning depth, and reduce edge-case failures.
- Predicted final loss range: **0.5-0.9** (from a starting point of ~1.5 if continuing from a prior LoRA, or ~2.5+ from base).
- Results vary by base model, pruning aggressiveness, and dataset quality.

---

## Section 1: Environment Setup

Install Unsloth (which handles MoE training efficiently with auto-detection of expert layers), TRL for SFT, and supporting libraries.

In [ ]:
# Install dependencies
# Unsloth handles MoE expert LoRA automatically — this is the key advantage
%pip install -q unsloth
%pip install -q --upgrade --force-reinstall --no-cache-dir torch triton
%pip install -q --no-deps cut_cross_entropy unsloth_zoo
%pip install -q sentencepiece protobuf datasets huggingface_hub hf_transfer
%pip install -q --no-deps trl peft accelerate bitsandbytes

# Speeds up HF downloads significantly for large models
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [ ]:
# Login to HuggingFace - needed for downloading models and pushing your trained LoRA
# Get a write token from https://huggingface.co/settings/tokens

from huggingface_hub import login

HF_TOKEN = "hf_YOUR_TOKEN_HERE"  # Replace with your token
login(token=HF_TOKEN)

---

## Section 2: Configuration

**Edit these for your specific model.** Defaults are tuned for REAP-40B-A3B class models on a single H200/H100.

In [ ]:
# ============================================================
# MODEL CONFIGURATION
# ============================================================

# The pruned MoE base model you want to repair
BASE_MODEL = "lovedheart/Qwen3-Coder-Next-REAP-40B-A3B"

# (Optional) Existing LoRA to continue from. Set to None for fresh training.
EXISTING_LORA = None  # e.g. "Johnblick187/Opus-4.7-Reasoning-REAP-40B-A3B-LoRA-v2"

# Where to save your new LoRA on HuggingFace
OUTPUT_LORA_NAME = "YourUsername/Your-Repaired-Model-LoRA"  # CHANGE THIS

# ============================================================
# DATA CONFIGURATION  
# ============================================================

# Datasets to mix. All must be in ShareGPT format ({conversations: [{from, value}, ...]})
DATASETS = [
    "Johnblick187/REAP-Repair-Reasoning-50K",  # 64K reasoning examples — the meat
    # Add more datasets here. Common additions:
    # "lordx64/reasoning-distill-opus-4-7-max-sft",  # ~7.8K Opus 4.7 distillations
    # "YourUsername/your-character-dataset",  # Your own task-specific data
]

# ============================================================
# TRAINING CONFIGURATION
# ============================================================

# Sequence length — 4096 is a good balance for reasoning data
# Reduce to 2048 if running out of VRAM
MAX_SEQ_LENGTH = 4096

# Learning rate. Lower if continuing from existing LoRA, higher for fresh training.
# - Fresh training on pruned base: 1e-4
# - Continuing from existing LoRA: 1e-5 to 5e-5
LEARNING_RATE = 1e-5 if EXISTING_LORA else 1e-4

# LoRA rank. Higher = more parameters trained = more capacity to learn
# 32-64 is sweet spot for repair work. 128+ if you want maximum recovery.
LORA_RANK = 64
LORA_ALPHA = 128  # Typically 2x rank

# How many epochs. 1 is usually enough — overfitting on reasoning data is real.
NUM_EPOCHS = 1

# Batch size. With 4096 ctx on H200, batch_size=1 + grad_accum=8 is safe.
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8

# Effective batch size = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS = 8

# Save checkpoint frequency. Every 100 steps = roughly every 30 min on H200.
SAVE_STEPS = 100

---

## Section 3: Load Model with MoE-Aware LoRA

Unsloth's killer feature for this workflow: **automatic detection of MoE expert layers and LoRA injection on `gate_up_proj` / `down_proj`**. Without this, you only train attention layers — which is way less effective for routing repair.

Watch the output for `Expert LoRA enabled` — that's the signal it's working.

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-detects (bf16 on Ampere+, fp16 elsewhere)
    load_in_4bit=True,  # 4-bit quantization for the base model — saves massive VRAM
    trust_remote_code=True,
)

print(f"\nBase model loaded: {BASE_MODEL}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Add LoRA adapters with MoE-aware target modules
# Unsloth auto-detects MoE structure and adds expert LoRA where it makes sense

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        # Attention
        "q_proj", "k_proj", "v_proj", "o_proj",
        # MLP (Unsloth auto-detects MoE and applies to expert layers)
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,  # 0 is optimal for Unsloth
    bias="none",  # 'none' is optimal for Unsloth
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized version
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

# Print trainable params — for a 40B MoE you should see 5-7B trainable
# That's the expert LoRA kicking in. If you see <500M trainable, expert LoRA didn't activate.
model.print_trainable_parameters()

In [ ]:
# (Optional) Continue from existing LoRA
# Skip this cell if EXISTING_LORA is None

if EXISTING_LORA:
    print(f"Loading existing LoRA: {EXISTING_LORA}")
    print("This continues training from the prior checkpoint instead of starting fresh.")
    
    # Load LoRA weights into the model
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, EXISTING_LORA, is_trainable=True)
    
    print("Existing LoRA loaded — training will continue from this point.")
else:
    print("Fresh training — no existing LoRA loaded.")

---

## Section 4: Setup Chat Template

Most modern Qwen / Llama / Mistral derivatives use ChatML or a close variant. Unsloth's `get_chat_template` handles the right formatting automatically.

In [ ]:
from unsloth.chat_templates import get_chat_template

# For Qwen3 family use "chatml". For Llama 3 use "llama-3". 
# Unsloth supports: chatml, llama-3, alpaca, gemma, mistral, phi-3, vicuna, zephyr, etc.
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",  # Adjust for your model family
)

# Quick sanity check that template works on a simple example
test_messages = [
    {"role": "system", "content": "You are helpful."},
    {"role": "user", "content": "What is 2+2?"},
    {"role": "assistant", "content": "2+2 equals 4."},
]
rendered = tokenizer.apply_chat_template(test_messages, tokenize=False)
print("Chat template renders to:")
print(rendered)

---

## Section 5: Load and Preprocess Datasets

We handle three concerns:

1. **Multiple datasets** — concat and shuffle
2. **Format standardization** — Unsloth's `standardize_sharegpt` normalizes different ShareGPT variants
3. **Nested `<think>` cleanup** — the REAP-Repair dataset has some examples with double `<think><think>` tags from the curation process. We fix that here.

In [ ]:
from datasets import load_dataset, concatenate_datasets
from unsloth.chat_templates import standardize_sharegpt
import re

loaded_datasets = []
for ds_name in DATASETS:
    print(f"Loading {ds_name}...")
    ds = load_dataset(ds_name, split="train")
    print(f"  Loaded {len(ds):,} examples")
    loaded_datasets.append(ds)

# Combine all datasets
if len(loaded_datasets) > 1:
    combined = concatenate_datasets(loaded_datasets)
else:
    combined = loaded_datasets[0]

print(f"\nCombined dataset: {len(combined):,} examples")

# Shuffle for training
combined = combined.shuffle(seed=42)

In [ ]:
# Standardize all variants of ShareGPT format to Unsloth's expected schema
# Handles: from/value vs role/content, different role names, etc.

combined = standardize_sharegpt(combined)
print(f"Standardized format. Sample structure:")
print(combined[0])

In [ ]:
# Fix nested <think><think> tags from the REAP-Repair curation process
# This is specific to that dataset; safe to run on others (no-op if no nested tags)

def fix_nested_think_tags(examples):
    convos = examples["conversations"]
    fixed = []
    for convo in convos:
        new_convo = []
        for turn in convo:
            content_key = "content" if "content" in turn else "value"
            content = turn.get(content_key, "")
            # Collapse multiple <think> openers into one
            content = re.sub(r'(<think>\s*){2,}', '<think>\n', content)
            # Collapse multiple </think> closers into one
            content = re.sub(r'(</think>\s*){2,}', '</think>\n', content)
            new_turn = {**turn, content_key: content}
            new_convo.append(new_turn)
        fixed.append(new_convo)
    return {"conversations": fixed}

combined = combined.map(fix_nested_think_tags, batched=True)
print("Cleaned nested <think> tags")

In [ ]:
# Format examples through the chat template
# Note: tokenize=False here gives us text strings; SFTTrainer will tokenize internally

def format_for_training(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in convos
    ]
    return {"text": texts}

combined = combined.map(format_for_training, batched=True)
print(f"\nFormatted for training. Sample:\n")
print(combined[0]["text"][:500])
print("...")

---

## Section 6: Training

Configuration tuned for stable MoE repair:

- **`paged_adamw_8bit`** — pages optimizer state to CPU RAM, critical for big models
- **`bf16`** — better numerical stability than fp16 for long training runs
- **`linear` LR scheduler with 50 warmup steps** — smooth ramp-up
- **Frequent checkpointing** — recover from crashes

**Before running the full train**, do a quick sanity check at the bottom of this section to make sure loss decreases on the first few steps.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=combined,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,  # Set True if you want to pack short examples — saves time but can confuse think blocks
    args=SFTConfig(
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        warmup_steps=50,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=False,  # bf16 is preferred for stability
        bf16=True,
        logging_steps=10,
        optim="paged_adamw_8bit",  # Critical for big MoE models — pages to CPU RAM
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="reap_repair_outputs",
        save_steps=SAVE_STEPS,
        save_total_limit=5,  # Keep last 5 checkpoints to avoid disk fill
        report_to="none",  # Set to "wandb" if you want tracking
    ),
)

In [ ]:
# Show GPU memory before training
import torch

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Pre-training VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Estimate training time
total_steps = (len(combined) * NUM_EPOCHS) // (PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS)
print(f"\nDataset size: {len(combined):,}")
print(f"Total training steps: {total_steps:,}")
print(f"Estimated time on H200: ~{total_steps / 1500:.1f} hours")
print(f"Estimated cost (at $3.50/hr): ${(total_steps / 1500) * 3.50:.2f}")

In [ ]:
# 🚀 START TRAINING
# This will take 4-8 hours depending on dataset size and GPU
# 
# Watch the loss output. It should:
# - Drop quickly in first 50 steps (warmup)
# - Settle into steady decrease through middle
# - Plateau toward the end
# 
# If loss INCREASES after step 100, stop and investigate — likely data issue.
# If loss is stuck flat, increase LR or check that expert LoRA is active.

trainer_stats = trainer.train()

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Training time: {trainer_stats.metrics['train_runtime'] / 3600:.2f} hours")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")
print(f"Total steps: {trainer_stats.global_step}")

---

## Section 7: Save and Push to HuggingFace

Save the final LoRA locally and push to HF for sharing.

In [ ]:
# Save locally first (always do this — guarantees you have it)
model.save_pretrained("reap_repair_final")
tokenizer.save_pretrained("reap_repair_final")
print("Saved locally to: reap_repair_final/")

In [ ]:
# Push to HuggingFace
# Make sure OUTPUT_LORA_NAME is set to YOUR username and a unique repo name

model.push_to_hub(OUTPUT_LORA_NAME, token=HF_TOKEN)
tokenizer.push_to_hub(OUTPUT_LORA_NAME, token=HF_TOKEN)

print(f"\n✅ Pushed to https://huggingface.co/{OUTPUT_LORA_NAME}")

---

## Section 8: Quick Inference Test

Before you celebrate, sanity-check that the trained model actually generates coherent output. If this is gibberish, something went wrong in training and you need to investigate before deploying.

In [ ]:
FastLanguageModel.for_inference(model)

# Test with a reasoning prompt
test_prompt = [
    {"role": "system", "content": "You are a thoughtful reasoning assistant. Think step-by-step."},
    {"role": "user", "content": "If a train leaves Boston at 3pm going 60mph and another leaves NYC at 4pm going 80mph toward Boston, and they're 200 miles apart, when do they meet?"},
]

inputs = tokenizer.apply_chat_template(
    test_prompt,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=2048,
    use_cache=True,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=False))

---

## Troubleshooting

### Loss is stuck around 2.0 and not dropping
- Check that expert LoRA is active: `model.print_trainable_parameters()` should show 4-7B trainable for a 40B MoE. If <500M, your target_modules don't include MoE expert layers.
- LR may be too low. Try 2e-4 for fresh training.

### OOM (Out of Memory)
- Drop `MAX_SEQ_LENGTH` from 4096 to 2048
- Make sure `optim="paged_adamw_8bit"` is set (pages to CPU RAM)
- Reduce LoRA rank from 64 to 32
- Make sure `load_in_4bit=True` for the base model

### Loss exploded after step 200
- Stop immediately. Some examples have malformed structure.
- Check your dataset: `print(combined[some_random_idx]["text"][-500:])` — look for truncation or weird characters
- Lower LR by 5x and resume from the last good checkpoint

### Model outputs gibberish during inference test
- Did the chat template match the model? Qwen wants chatml, Llama wants llama-3, etc.
- Try `add_generation_prompt=True` (for inference) vs `False` (for training data)
- The first few output tokens should be coherent — if they're random tokens, generation params are broken, not the model

### Training is way slower than expected
- Check that `use_gradient_checkpointing="unsloth"` is set (NOT `True` or `False`)
- Check `packing=False` — packing can slow things down for variable-length data
- VRAM bottleneck shows up as low GPU util (<70%) — reduce seq length

---

## Credits

- Notebook by [Johnblick187](https://huggingface.co/Johnblick187)
- Powered by [Unsloth](https://github.com/unslothai/unsloth) for MoE-aware LoRA
- REAP technique by [Cerebras Systems](https://arxiv.org/) — original pruning approach
- Default dataset: [REAP-Repair-Reasoning-50K](https://huggingface.co/datasets/Johnblick187/REAP-Repair-Reasoning-50K)

If this notebook helps you, consider:
- ❤️ Liking the dataset on HF
- Sharing your trained LoRA back to the community
- Reporting any issues or improvements you find

Happy training. 🔧